# Inverse covariance estimation with MFCF-LoGo.

Sanity-check MFCFLoGo against GraphicalLassoCV on synthetic data whose
true precision is *not* chordal — in both the well-posed regime (n > p)
and the high-dimensional regime (n << p) MFCF is designed for.

Each scenario also saves a side-by-side heatmap of the ground-truth
precision and every estimator's reconstruction to ``./figs/``.

Scenarios
---------
* ``scenario_er``                  Erdős-Rényi sparse precision, Gaussian.
* ``scenario_cycle``               Length-p cycle + a few chords, Gaussian.
* ``scenario_blocks_gaussian``     Block-diagonal precision, Gaussian.
* ``scenario_blocks_nongaussian``  Non-Gaussian latent-block features (cos/
                                   sin/quadratic of a shared latent).
* ``scenario_mi_collapse``         Sample-rich Gaussian — both Linfoot MI
                                   estimators (KSG and histogram) collapse
                                   to ``|Pearson rho|`` and all three MFCF
                                   paths become identical.
The first four scenarios run in two regimes: ``n > p`` and ``n << p``.
Every mutual-information path is evaluated with *both* estimators:
``mi_estimator="ksg"`` (k-NN) and ``mi_estimator="histogram"`` (the fast
equal-frequency plug-in estimator, built for the ``n << p`` regime).

Design rationale
----------------
The three Gaussian generators form a deliberate triangle of difficulty,
chosen so that no single graph property silently advantages MFCF:

* **ER** is the *neutral* baseline — generic random sparse precision
  with no exploitable structure, almost-surely non-chordal once
  ``p * edge_prob >= 2``. It is also the standard benchmark in the
  GLasso / neighbourhood-selection literature, so it tests MFCF on
  its competitor's home turf. The single ``edge_prob`` knob lets the
  same generator probe both the ``n > p`` and ``n << p`` regimes.
* **Cycle + chords** is the *adversarial* target: a length-``p`` cycle
  is the canonical non-chordal graph for ``p >= 4``. MFCF can only
  approximate it via a chordal cover, so this scenario quantifies the
  unavoidable approximation cost — it is the worst case for any
  chordal-by-construction estimator.
* **Block-diagonal** is the *friendly* target: chordal within each
  block, hard conditional independence across blocks. MFCF is
  expected to match or beat GLasso here.

``scenario_blocks_nongaussian`` is the only place where MFCF-MI is
*supposed* to beat MFCF-corr: the latent block-mates are linked by
non-monotone maps (``cos(2z)``, ``sin(2z)``, ``z^2 - 1``, ``|z| - 0.8``),
which drive Pearson correlation to near zero while leaving the
mutual information large.

``scenario_mi_collapse`` is the dual control: a sample-rich Gaussian
where, by Linfoot's 1957 identity, both normalised MI estimators
(KSG and histogram) must converge to ``|Pearson rho|``. With enough
samples the three MFCF paths *must* land on the same edges; the
scenario quantifies how tight that collapse really is at finite n.

In [1]:
import os
import time
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import optuna
from optuna.samplers import TPESampler
from scipy import linalg
from sklearn.covariance import GraphicalLassoCV, empirical_covariance, log_likelihood
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import KFold

from mfcf_logo import MFCFLoGo
from mutual_information import mutual_information_matrix

### Define global hyperparameters

In [2]:
optuna.logging.set_verbosity(optuna.logging.ERROR)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)

HPO_TIME_BUDGET = 30.0   # seconds, per (scenario, similarity)
# Extended-BIC parameter gamma (Foygel & Drton, 2010).  gamma=0 is classical
# BIC; gamma=0.5 is the recommended high-dimensional value.  "auto" follows
# their regime guidance: gamma=0 when n > p (BIC is consistent there) and
# gamma=0.5 when p >= n (the extended penalty is needed).
HPO_EBIC_GAMMA  = "auto"
# Reject trials whose (unregularized) MFCF-LoGo precision is ill-conditioned:
# a likelihood criterion can otherwise be gamed by a near-singular precision
# with an inflated log-determinant.  GraphicalLasso avoids this via its L1
# penalty; this guard enforces the same positive-definite well-posedness.
HPO_MAX_CONDITION = 1e8   # max allowed condition number of the precision
HPO_TPE_STARTUP = 12      # random trials before the TPE surrogate kicks in
HPO_SEED        = 42

### Define the data generators

In [3]:
def generate_er_precision(p, edge_prob, prng,
                          weight_scale=0.3, diag_margin=0.5):
    """Erdős-Rényi off-diagonal pattern; diagonally dominant for SPD.

    Role
    ----
    Neutral baseline. No block / chordal / cycle structure is imposed,
    so the resulting graph is almost-surely non-chordal once
    ``p * edge_prob >= 2``. This is the canonical sparse-precision
    benchmark from the GLasso literature — putting it here means GLasso
    is being compared on its home turf. Diagonal dominance
    (``|row sum| + margin``) is the cheap way to guarantee SPD without
    constraining the off-diagonal pattern.
    """
    mask = np.triu(prng.uniform(size=(p, p)) < edge_prob, k=1)
    P = np.zeros((p, p))
    P[mask] = prng.uniform(-weight_scale, weight_scale, size=int(mask.sum()))
    P = P + P.T
    np.fill_diagonal(P, np.abs(P).sum(axis=1) + diag_margin)
    return P


def generate_cycle_plus_chords_precision(p, n_chords, prng,
                                         weight_scale=0.3, diag_margin=0.5):
    """Length-p cycle plus a few extra chords — non-chordal for p >= 4.

    Role
    ----
    Adversarial target for any chordal-by-construction estimator.
    A pure cycle of length ``p >= 4`` is the textbook non-chordal
    graph: every chordal cover MFCF can emit must either add fill-in
    edges or drop true ones. The handful of extra chords keeps the
    structure non-trivial without making it chordal. This scenario
    measures the *unavoidable* approximation cost of forcing the
    estimator into a clique forest.
    """
    P = np.zeros((p, p))
    for i in range(p):
        j = (i + 1) % p
        w = prng.uniform(-weight_scale, weight_scale)
        P[i, j] = w
        P[j, i] = w
    placed = 0
    while placed < n_chords:
        i, j = prng.randint(0, p, size=2)
        if i == j or P[i, j] != 0 or abs(i - j) <= 1 or abs(i - j) == p - 1:
            continue
        w = prng.uniform(-weight_scale, weight_scale)
        P[i, j] = w
        P[j, i] = w
        placed += 1
    np.fill_diagonal(P, np.abs(P).sum(axis=1) + diag_margin)
    return P


def generate_block_precision(n_blocks, block_size, prng,
                             weight_scale=0.3, diag_margin=0.5):
    """Block-diagonal sparse precision: every pair of variables in distinct
    blocks is conditionally independent (hard zero in the precision).
    Within each block, edge weights are i.i.d. uniform.

    Role
    ----
    Friendly target. Each block is a complete sub-graph (chordal),
    blocks are conditionally independent, and the union is a forest
    of cliques by construction — exactly the family MFCF is built to
    recover. If MFCF does not match GLasso here, something is wrong.
    """
    p = n_blocks * block_size
    P = np.zeros((p, p))
    tri_mask = np.triu(np.ones((block_size, block_size), dtype=bool), k=1)
    n_block_edges = int(tri_mask.sum())
    for k in range(n_blocks):
        i0 = k * block_size
        i1 = i0 + block_size
        W = np.zeros((block_size, block_size))
        W[tri_mask] = prng.uniform(-weight_scale, weight_scale,
                                   size=n_block_edges)
        W = W + W.T
        P[i0:i1, i0:i1] = W
    np.fill_diagonal(P, np.abs(P).sum(axis=1) + diag_margin)
    return P


def _generate_nonmonotone_blocks(n, n_latents, block_size, seed):
    """Latent-block non-Gaussian features — MI should beat correlation here.

    Role
    ----
    The only scenario where MFCF-MI is *expected* to dominate
    MFCF-corr. Block-mates share a latent ``Z`` but are observed
    through non-monotone (and even-symmetric) maps such as
    ``cos(2z)`` and ``z^2 - 1``. Pearson correlation between two
    such columns is near zero — symmetric noise around the same
    centre — while mutual information remains large because the
    columns are deterministic functions of the same source. This
    is the case-study justifying the MI gain path at all.
    """
    prng = np.random.RandomState(seed)
    p = n_latents * block_size
    Z = prng.randn(n, n_latents)
    fs = [
        lambda z: np.cos(2 * z),
        lambda z: np.sin(2 * z),
        lambda z: z ** 2 - 1.0,
        lambda z: np.abs(z) - 0.8,
    ]
    X = np.empty((n, p))
    true_block = np.empty(p, dtype=int)
    for k in range(n_latents):
        for b in range(block_size):
            fn = fs[b % len(fs)]
            X[:, k * block_size + b] = fn(Z[:, k]) + 0.05 * prng.randn(n)
            true_block[k * block_size + b] = k
    prec_true = (true_block[:, None] == true_block[None, :]).astype(float)
    return X, prec_true


def _standardise(prec):
    """Move (cov, prec) to the unit-diagonal-covariance parametrisation.

    Sampling and edge-set comparisons are invariant under per-feature
    rescaling, so we always evaluate in the canonical form where
    ``diag(Sigma) == 1``. Without this, large random diagonal entries
    in the precision would dominate the per-panel colour scale and
    visually wash out the off-diagonal pattern we actually care about.
    """
    cov = linalg.inv(prec)
    d = np.sqrt(np.diag(cov))
    cov = cov / d / d[:, None]
    prec = prec * d * d[:, None]
    return cov, prec

### Define the scoring functions

In [4]:
def _edges_from_precision(P, thr=1e-8):
    A = np.abs(P) > thr
    np.fill_diagonal(A, False)
    p = P.shape[0]
    return {(i, j) for i in range(p) for j in range(i + 1, p) if A[i, j]}


def _edge_scores(E_est, E_true):
    tp = len(E_est & E_true)
    fp = len(E_est - E_true)
    fn = len(E_true - E_est)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    return prec, rec, f1


def _evaluate(name, model, X, E_true):
    t0 = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        warnings.simplefilter("ignore", RuntimeWarning)
        model.fit(X)
    dt = time.time() - t0
    p, r, f = _edge_scores(_edges_from_precision(model.precision_), E_true)
    print(f"  {name:28s}  P={p:.3f}  R={r:.3f}  F1={f:.3f}  ({dt:.2f}s)")
    return model.precision_.copy()

### Hyperparameter tuning

In [5]:
def _mfcf_hpo(
    X: np.ndarray,
    similarity: str,
    mi_estimator: str = "ksg",
    *,
    selection: str = "ebic",
    time_budget:      float = HPO_TIME_BUDGET,
    ebic_gamma=HPO_EBIC_GAMMA,
    cv: int = 5,
    max_condition: float = HPO_MAX_CONDITION,
    n_startup_trials: int = HPO_TPE_STARTUP,
    seed:             int = HPO_SEED,
) -> tuple[MFCFLoGo, dict, int, dict]:
    """Select :class:`MFCFLoGo` hyper-parameters, leakage-free.

    Two selection criteria are available via ``selection``:

    ``"ebic"`` (default)
        Minimise the Extended Bayesian Information Criterion for Gaussian
        graphical models (Foygel & Drton, NIPS 2010):

            EBIC_gamma(E) = -2 * l_n(Theta_hat_E) + |E| * log(n)
                            + 4 * gamma * |E| * log(p),   gamma in [0, 1].

        MFCF returns a *decomposable* (chordal) graph, so the MFCF-LoGo
        precision is exactly the decomposable Gaussian MLE (Lauritzen 1996,
        Prop. 5.9) and ``l_n`` is the true maximised likelihood -- the criterion
        is exact, evaluated in-sample on the full ``X``.  ``gamma="auto"`` (the
        default) follows Foygel & Drton's regime guidance: ``gamma=0``
        (classical BIC) when ``n > p`` and ``gamma=0.5`` when ``p >= n``.

    ``"cv_loglik"``
        Maximise the K-fold cross-validated held-out Gaussian log-likelihood --
        *exactly the outer criterion* :class:`sklearn.covariance.GraphicalLassoCV`
        uses to choose its penalty.  Provided so MFCF can be compared to
        GraphicalLassoCV under a matched selection criterion (isolating the
        estimator from the model-selection rule).

    The ground truth is never used.  Returns the model refit on the full ``X``
    with the winning parameters, the parameter dict, the trial count, and a
    diagnostics dict (``selection``, ``best_value`` and, for EBIC, ``gamma``).
    """
    if selection not in ("ebic", "cv_loglik"):
        raise ValueError(f"selection must be 'ebic' or 'cv_loglik', got {selection!r}.")
    n_samples, n_features = X.shape
    # Cap clique size at n-1: the decomposable Gaussian MLE (hence both the EBIC
    # likelihood and a non-degenerate CV fit) is only defined when every clique
    # submatrix of the sample covariance is invertible, i.e. clique size <= n-1.
    max_cs_upper = max(2, min(n_features - 1, n_samples - 1))
    iu = np.triu_indices(n_features, k=1)
    log_n = float(np.log(n_samples))
    log_p = float(np.log(n_features))
    if ebic_gamma == "auto":
        gamma = 0.0 if n_samples > n_features else 0.5
    else:
        gamma = float(ebic_gamma)

    minimise = (selection == "ebic")
    infeasible = float("inf") if minimise else float("-inf")

    def _ill_conditioned(P):
        # Reject non-PD or near-singular precisions whose inflated log-det
        # would game the likelihood (the unregularized-MLE analogue of the
        # positive-definiteness GLasso's L1 penalty guarantees).
        ev = np.linalg.eigvalsh(P)
        return ev[0] <= 0.0 or (ev[-1] / ev[0]) > max_condition

    def _build(threshold, min_cs, max_cs, coord_num, mi_k, mi_bins, mi_norm):
        return MFCFLoGo(
            threshold=threshold,
            min_clique_size=min_cs,
            max_clique_size=max_cs,
            coordination_number=coord_num,
            similarity=similarity,
            mi_estimator=mi_estimator,
            mi_n_neighbors=mi_k,
            mi_n_bins=mi_bins,
            mi_normalize=mi_norm,
            mi_random_state=seed,
        )

    def _emp_secondmoment(Xs):
        # Match the scale on which the precision was built: correlation for the
        # correlation similarity, covariance for the MI similarities.
        if similarity == "correlation":
            return np.corrcoef(Xs, rowvar=False)
        return empirical_covariance(Xs, assume_centered=False)

    def objective(trial: optuna.Trial) -> float:
        threshold = trial.suggest_float("threshold", 0.0, 0.8)
        max_cs_raw = trial.suggest_int("max_clique_size", 2, max_cs_upper)
        min_cs_raw = trial.suggest_int("min_clique_size", 1, 5)
        min_cs = min(min_cs_raw, max_cs_raw)
        max_cs = max(min_cs_raw, max_cs_raw)
        coord_num = trial.suggest_int(
            "coordination_number", 1, max(2, n_features), log=True,
        )
        if similarity == "mutual_information":
            mi_norm = trial.suggest_categorical("mi_normalize",
                                                ["linfoot", "none"])
            if mi_estimator == "histogram":
                mi_k    = 3
                mi_bins = trial.suggest_int("mi_n_bins", 2, 12)
            else:
                mi_k    = trial.suggest_int("mi_n_neighbors", 2, 20)
                mi_bins = "auto"
        else:
            mi_k, mi_bins, mi_norm = 3, "auto", "linfoot"

        if selection == "ebic":
            est = _build(threshold, min_cs, max_cs, coord_num, mi_k, mi_bins, mi_norm)
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    est.fit(X)
            except Exception:
                return infeasible
            P = est.precision_
            if not np.all(np.isfinite(P)) or _ill_conditioned(P):
                return infeasible
            loglik = n_samples * float(log_likelihood(_emp_secondmoment(X), P))
            n_edges = int((np.abs(P[iu]) > 1e-8).sum())
            ebic = (-2.0 * loglik + n_edges * log_n
                    + 4.0 * gamma * n_edges * log_p)
            return ebic if np.isfinite(ebic) else infeasible

        # selection == "cv_loglik": same outer criterion as GraphicalLassoCV.
        kf = KFold(n_splits=cv, shuffle=True, random_state=seed)
        total, nfold = 0.0, 0
        for tr_idx, te_idx in kf.split(X):
            est = _build(threshold, min_cs, max_cs, coord_num, mi_k, mi_bins, mi_norm)
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    est.fit(X[tr_idx])
            except Exception:
                return infeasible
            P = est.precision_
            if not np.all(np.isfinite(P)) or _ill_conditioned(P):
                return infeasible
            ll = float(log_likelihood(_emp_secondmoment(X[te_idx]), P))
            if not np.isfinite(ll):
                return infeasible
            total += ll
            nfold += 1
        return total / nfold

    sampler = TPESampler(
        seed=seed,
        multivariate=True,
        group=True,
        n_startup_trials=n_startup_trials,
    )
    study = optuna.create_study(
        direction="minimize" if minimise else "maximize", sampler=sampler,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        study.optimize(objective, timeout=time_budget, n_jobs=1,
                       show_progress_bar=False)

    feasible = [t for t in study.trials
                if t.value is not None and np.isfinite(t.value)]
    if not feasible:
        warnings.warn(
            f"HPO ({similarity}, {selection}) found no feasible trial; "
            f"refitting with MFCFLoGo defaults.",
            RuntimeWarning,
        )
        best = dict(
            threshold=0.0,
            min_clique_size=1,
            max_clique_size=min(4, max_cs_upper),
            coordination_number=int(max(2, n_features)),
        )
        diagnostics = dict(selection=selection, best_value=infeasible, gamma=gamma)
    else:
        best_trial = (min if minimise else max)(feasible, key=lambda t: t.value)
        best = dict(best_trial.params)
        diagnostics = dict(selection=selection,
                           best_value=float(best_trial.value), gamma=gamma)

    min_cs_raw = best.get("min_clique_size", 1)
    max_cs_raw = best.get("max_clique_size", min(4, max_cs_upper))
    best_min_cs = min(min_cs_raw, max_cs_raw)
    best_max_cs = max(min_cs_raw, max_cs_raw)
    final = MFCFLoGo(
        threshold=best.get("threshold", 0.0),
        min_clique_size=best_min_cs,
        max_clique_size=best_max_cs,
        coordination_number=best.get("coordination_number",
                                     int(max(2, n_features))),
        similarity=similarity,
        mi_estimator=mi_estimator,
        mi_n_neighbors=best.get("mi_n_neighbors", 3),
        mi_n_bins=best.get("mi_n_bins", "auto"),
        mi_normalize=best.get("mi_normalize", "linfoot"),
        mi_random_state=seed,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        final.fit(X)
    return final, best, len(study.trials), diagnostics


def _evaluate_mfcf_hpo(name, X, similarity, E_true,
                       time_budget=HPO_TIME_BUDGET, mi_estimator="ksg",
                       selection="ebic"):
    """Run the leakage-free HPO and report edge-set scores.

    Only the refit precision is scored against ``E_true`` (outside the HPO
    loop); Optuna never sees ``E_true``.
    """
    t0 = time.time()
    final, best, n_trials, diag = _mfcf_hpo(
        X, similarity=similarity, mi_estimator=mi_estimator,
        selection=selection, time_budget=time_budget,
    )
    dt = time.time() - t0
    P = final.precision_
    p, r, f = _edge_scores(_edges_from_precision(P), E_true)
    edges = len(_edges_from_precision(P))
    cs_lo = min(best.get("min_clique_size", 1), best.get("max_clique_size", 1))
    cs_hi = max(best.get("min_clique_size", 1), best.get("max_clique_size", 1))
    if diag["selection"] == "ebic":
        crit = f"gamma={diag['gamma']:.1f}, EBIC={diag['best_value']:.1f}"
    else:
        crit = f"CV-LL={diag['best_value']:.3f}"
    print(f"  {name:28s}  P={p:.3f}  R={r:.3f}  F1={f:.3f}  "
          f"({dt:.1f}s, {n_trials} trials, "
          f"clique=[{cs_lo},{cs_hi}], thr={best.get('threshold', 0.0):.3f}, "
          f"edges={edges}, {crit})")
    return P.copy()

### Define the plotting functions

In [6]:
def _zero_diag(P):
    P = P.copy()
    np.fill_diagonal(P, 0.0)
    return P


def _panel_vmax(M):
    """99th-percentile of non-zero |entries|, with safe fallback."""
    absM = np.abs(M)
    nz = absM[absM > 0]
    if nz.size == 0:
        return 1.0
    v = float(np.quantile(nz, 0.99))
    return v if np.isfinite(v) and v > 0 else float(absM.max() or 1.0)


def _save_comparison(P_true, panels, outfile, suptitle):
    """``panels`` is a list of ``(title, precision_matrix)``.

    The diagonal is masked out so the visualisation focuses on the
    conditional-dependence structure rather than the unit diagonal.
    Each panel uses its own colour scale (clipped to the 99th percentile
    of non-zero entries) because MFCF and GLasso precisions can live at
    wildly different magnitudes — a shared scale washes one out.
    """
    matrices = [_zero_diag(P_true)] + [_zero_diag(P) for _, P in panels]
    titles = ["Ground truth"] + [t for t, _ in panels]
    n = len(matrices)
    fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 3.8))
    for ax, title, M in zip(axes, titles, matrices):
        vmax = _panel_vmax(M)
        im = ax.imshow(M, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                       interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, shrink=0.75, fraction=0.046, pad=0.02)
    fig.suptitle(suptitle, fontsize=11)
    fig.savefig(outfile, dpi=120, bbox_inches="tight")
    plt.close(fig)

### Define the well-posed $(n > p)$ scenarios

In [7]:
def scenario_er(seed=1):
    p, n = 60, 400
    prng = np.random.RandomState(seed)
    prec = generate_er_precision(p, edge_prob=0.06, prng=prng)
    cov, prec = _standardise(prec)
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[ER precision, n>p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_mfcf = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI-ksg  HPO", X, "mutual_information", E_true)
    P_mih  = _evaluate_mfcf_hpo("MFCF MI-hist HPO", X, "mutual_information", E_true,
                                mi_estimator="histogram")
    # Same outer selection criterion as GraphicalLassoCV (K-fold CV held-out
    # log-likelihood) applied to the MFCF HPO -> criterion-matched comparison.
    _evaluate_mfcf_hpo("MFCF corr  CV-LL", X, "correlation",        E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-ksg  CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-hist CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik", mi_estimator="histogram")
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_mfcf), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "er_lowdim.png"),
        f"Erdős-Rényi precision  (p={p}, n={n})",
    )
    '''


def scenario_cycle(seed=1):
    p, n = 40, 300
    prng = np.random.RandomState(seed)
    prec = generate_cycle_plus_chords_precision(p, n_chords=5, prng=prng)
    cov, prec = _standardise(prec)
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[Cycle + chords, n>p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_mfcf = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI-ksg  HPO", X, "mutual_information", E_true)
    P_mih  = _evaluate_mfcf_hpo("MFCF MI-hist HPO", X, "mutual_information", E_true,
                                mi_estimator="histogram")
    # Same outer selection criterion as GraphicalLassoCV (K-fold CV held-out
    # log-likelihood) applied to the MFCF HPO -> criterion-matched comparison.
    _evaluate_mfcf_hpo("MFCF corr  CV-LL", X, "correlation",        E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-ksg  CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-hist CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik", mi_estimator="histogram")
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_mfcf), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "cycle_lowdim.png"),
        f"Cycle + chords  (p={p}, n={n})",
    )
    '''


def scenario_blocks_gaussian(seed=1):
    n_blocks, block_size, n = 5, 4, 400
    prng = np.random.RandomState(seed)
    prec = generate_block_precision(n_blocks, block_size, prng)
    cov, prec = _standardise(prec)
    p = n_blocks * block_size
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[Block-diag, Gaussian, n>p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_mfcf = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI-ksg  HPO", X, "mutual_information", E_true)
    P_mih  = _evaluate_mfcf_hpo("MFCF MI-hist HPO", X, "mutual_information", E_true,
                                mi_estimator="histogram")
    # Same outer selection criterion as GraphicalLassoCV (K-fold CV held-out
    # log-likelihood) applied to the MFCF HPO -> criterion-matched comparison.
    _evaluate_mfcf_hpo("MFCF corr  CV-LL", X, "correlation",        E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-ksg  CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-hist CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik", mi_estimator="histogram")
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_mfcf), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "blocks_gaussian_lowdim.png"),
        f"Block-diag precision, Gaussian  (p={p}, n={n})",
    )
    '''


def scenario_blocks_nongaussian(seed=1):
    n_latents, block_size, n = 5, 4, 600
    X, prec_true = _generate_nonmonotone_blocks(n, n_latents, block_size, seed)
    E_true = _edges_from_precision(prec_true)
    p = n_latents * block_size
    print(f"\n[Non-Gaussian blocks, n>p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_corr = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI-ksg  HPO", X, "mutual_information", E_true)
    P_mih  = _evaluate_mfcf_hpo("MFCF MI-hist HPO", X, "mutual_information", E_true,
                                mi_estimator="histogram")
    # Same outer selection criterion as GraphicalLassoCV (K-fold CV held-out
    # log-likelihood) applied to the MFCF HPO -> criterion-matched comparison.
    _evaluate_mfcf_hpo("MFCF corr  CV-LL", X, "correlation",        E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-ksg  CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-hist CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik", mi_estimator="histogram")
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec_true,
        [("MFCF corr HPO", P_corr), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "blocks_nongaussian_lowdim.png"),
        f"Non-Gaussian blocks  (p={p}, n={n})",
    )
    '''

### Define the high-dimensional scenarios $(n << p)$

In [8]:
def scenario_er_highdim(seed=1):
    p, n = 150, 50
    prng = np.random.RandomState(seed)
    prec = generate_er_precision(p, edge_prob=0.025, prng=prng)
    cov, prec = _standardise(prec)
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[ER precision, n<<p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_mfcf = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI-ksg  HPO", X, "mutual_information", E_true)
    P_mih  = _evaluate_mfcf_hpo("MFCF MI-hist HPO", X, "mutual_information", E_true,
                                mi_estimator="histogram")
    # Same outer selection criterion as GraphicalLassoCV (K-fold CV held-out
    # log-likelihood) applied to the MFCF HPO -> criterion-matched comparison.
    _evaluate_mfcf_hpo("MFCF corr  CV-LL", X, "correlation",        E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-ksg  CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-hist CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik", mi_estimator="histogram")
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_mfcf), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "er_highdim.png"),
        f"Erdős-Rényi precision  (p={p}, n={n}, n<<p)",
    )
    '''


def scenario_cycle_highdim(seed=1):
    p, n = 80, 30
    prng = np.random.RandomState(seed)
    prec = generate_cycle_plus_chords_precision(p, n_chords=8, prng=prng)
    cov, prec = _standardise(prec)
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[Cycle + chords, n<<p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_mfcf = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI-ksg  HPO", X, "mutual_information", E_true)
    P_mih  = _evaluate_mfcf_hpo("MFCF MI-hist HPO", X, "mutual_information", E_true,
                                mi_estimator="histogram")
    # Same outer selection criterion as GraphicalLassoCV (K-fold CV held-out
    # log-likelihood) applied to the MFCF HPO -> criterion-matched comparison.
    _evaluate_mfcf_hpo("MFCF corr  CV-LL", X, "correlation",        E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-ksg  CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-hist CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik", mi_estimator="histogram")
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_mfcf), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "cycle_highdim.png"),
        f"Cycle + chords  (p={p}, n={n}, n<<p)",
    )
    '''


def scenario_blocks_gaussian_highdim(seed=1):
    n_blocks, block_size, n = 6, 8, 40
    prng = np.random.RandomState(seed)
    prec = generate_block_precision(n_blocks, block_size, prng)
    cov, prec = _standardise(prec)
    p = n_blocks * block_size
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[Block-diag, Gaussian, n<<p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_mfcf = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI-ksg  HPO", X, "mutual_information", E_true)
    P_mih  = _evaluate_mfcf_hpo("MFCF MI-hist HPO", X, "mutual_information", E_true,
                                mi_estimator="histogram")
    # Same outer selection criterion as GraphicalLassoCV (K-fold CV held-out
    # log-likelihood) applied to the MFCF HPO -> criterion-matched comparison.
    _evaluate_mfcf_hpo("MFCF corr  CV-LL", X, "correlation",        E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-ksg  CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-hist CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik", mi_estimator="histogram")
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_mfcf), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "blocks_gaussian_highdim.png"),
        f"Block-diag precision, Gaussian  (p={p}, n={n}, n<<p)",
    )
    '''


def scenario_blocks_nongaussian_highdim(seed=1):
    n_latents, block_size, n = 6, 8, 40
    X, prec_true = _generate_nonmonotone_blocks(n, n_latents, block_size, seed)
    E_true = _edges_from_precision(prec_true)
    p = n_latents * block_size
    print(f"\n[Non-Gaussian blocks, n<<p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_corr = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI-ksg  HPO", X, "mutual_information", E_true)
    P_mih  = _evaluate_mfcf_hpo("MFCF MI-hist HPO", X, "mutual_information", E_true,
                                mi_estimator="histogram")
    # Same outer selection criterion as GraphicalLassoCV (K-fold CV held-out
    # log-likelihood) applied to the MFCF HPO -> criterion-matched comparison.
    _evaluate_mfcf_hpo("MFCF corr  CV-LL", X, "correlation",        E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-ksg  CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik")
    _evaluate_mfcf_hpo("MFCF MI-hist CV-LL", X, "mutual_information", E_true,
                       selection="cv_loglik", mi_estimator="histogram")
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec_true,
        [("MFCF corr HPO", P_corr), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "blocks_nongaussian_highdim.png"),
        f"Non-Gaussian blocks  (p={p}, n={n}, n<<p)",
    )
    '''

### Define the "collapse" scenario

In [9]:
def scenario_mi_collapse(seed=1):
    """All three similarities coincide under joint Gaussianity.

    By Linfoot's 1957 identity the Linfoot-normalised mutual information
    equals ``|Pearson rho|`` for jointly Gaussian variables, so with enough
    samples both MI estimators -- the KSG k-NN estimator and the histogram
    (plug-in) estimator -- converge to ``|Pearson rho|`` and all three MFCF
    paths become indistinguishable.

    This scenario quantifies the collapse: it reports the elementwise
    distance among the three similarity matrices and shows the three MFCF
    precisions land at the same edges.

    Each similarity is tuned INDEPENDENTLY over the full MFCF hyperparameter
    set: under joint Gaussianity the three similarity matrices coincide, so
    their HPO objective landscapes coincide and the searches should converge
    to the same optimum -- i.e. the three precisions should land on the same
    edges.  (Residual differences reflect HPO stochasticity / unequal trial
    counts -- the slow KSG estimator gets fewer trials per wall-clock second
    -- not the similarities, which are interchangeable here.)
    """
    # Construct cov directly: 3 equi-correlated blocks of size 4 with
    # within-block rho=0.7.  This gives sizeable signal correlations,
    # so the KSG bias does not dominate at finite n.
    n_blocks, block_size, n = 3, 4, 10000
    p = n_blocks * block_size
    rho = 0.7
    cov = np.eye(p)
    for k in range(n_blocks):
        i0 = k * block_size
        i1 = i0 + block_size
        cov[i0:i1, i0:i1] = (1.0 - rho) * np.eye(block_size) + rho
    prec = linalg.inv(cov)
    prng = np.random.RandomState(seed)
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[MI collapse to |Pearson|]  p={p}, n={n}, |E_true|={len(E_true)}")

    # Three similarity matrices: Pearson |rho|, Linfoot-KSG MI and the new
    # Linfoot-histogram MI.  Under joint Gaussianity all three coincide.
    C_corr = np.abs(np.corrcoef(X, rowvar=False))
    C_ksg  = mutual_information_matrix(
        X, estimator="ksg", n_neighbors=3, normalize="linfoot",
        random_state=seed)
    C_hist = mutual_information_matrix(
        X, estimator="histogram", normalize="linfoot")
    for C in (C_corr, C_ksg, C_hist):
        np.fill_diagonal(C, 0.0)

    def _pair(a, b):
        d = np.abs(a - b)
        r = float(np.corrcoef(a.ravel(), b.ravel())[0, 1])
        return d.max(), d.mean(), r

    print("  similarity-matrix collapse (off-diagonal):")
    print("    pair               max-abs   mean-abs   Pearson")
    for label, a, b in (("KSG  vs |corr|", C_ksg,  C_corr),
                        ("hist vs |corr|", C_hist, C_corr),
                        ("hist vs KSG   ", C_hist, C_ksg)):
        mx, mn, r = _pair(a, b)
        print(f"    {label:16s} {mx:7.4f}   {mn:7.4f}   {r:7.4f}")

    # Independently HPO each similarity over the FULL hyperparameter set,
    # then compare the resulting precisions.  Each call optimises threshold,
    # clique-size bounds, coordination number, mi_normalize and its own
    # estimator knob (mi_n_neighbors for KSG / mi_n_bins for histogram).
    P_corr = _evaluate_mfcf_hpo("MFCF corr     HPO", X, "correlation",
                                E_true)
    P_ksg  = _evaluate_mfcf_hpo("MFCF MI-ksg   HPO", X, "mutual_information",
                                E_true, mi_estimator="ksg")
    P_hist = _evaluate_mfcf_hpo("MFCF MI-hist  HPO", X, "mutual_information",
                                E_true, mi_estimator="histogram")
    E_corr = _edges_from_precision(P_corr)
    E_ksg  = _edges_from_precision(P_ksg)
    E_hist = _edges_from_precision(P_hist)

    def _jac(a, b):
        return len(a & b) / max(len(a | b), 1)

    # Edge-set Jaccard is the scale-invariant collapse metric.  (We do NOT
    # compare precision *values* across paths: the correlation path builds a
    # correlation-scale precision and the MI paths a covariance-scale one,
    # so |P_corr - P_mi| would not be apples-to-apples.)
    print("  precision-output collapse (edge-set Jaccard; scale-invariant):")
    print(f"    Jaccard(E_corr, E_ksg)            = {_jac(E_corr, E_ksg):.4f}")
    print(f"    Jaccard(E_corr, E_hist)           = {_jac(E_corr, E_hist):.4f}")
    print(f"    Jaccard(E_ksg,  E_hist)           = {_jac(E_ksg,  E_hist):.4f}")

    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_corr), ("MFCF MI-ksg HPO", P_ksg),
         ("MFCF MI-hist HPO", P_hist)],
        os.path.join(FIG_DIR, "mi_collapse.png"),
        f"MI collapse to |Pearson|, Gaussian  (p={p}, n={n})",
    )
    '''

### Running tests

In [10]:
scenario_er()


[ER precision, n>p]  p=60, n=400, |E_true|=114
  MFCF corr  HPO                P=0.895  R=0.447  F1=0.596  (30.2s, 2466 trials, clique=[1,37], thr=0.017, edges=57, gamma=0.0, EBIC=67466.4)
  MFCF MI-ksg  HPO              P=0.500  R=0.167  F1=0.250  (30.3s, 256 trials, clique=[1,22], thr=0.061, edges=38, gamma=0.0, EBIC=67707.7)
  MFCF MI-hist HPO              P=0.587  R=0.325  F1=0.418  (30.2s, 2179 trials, clique=[2,11], thr=0.015, edges=63, gamma=0.0, EBIC=67517.2)
  MFCF corr  CV-LL              P=0.973  R=0.316  F1=0.477  (30.1s, 846 trials, clique=[1,37], thr=0.031, edges=37, CV-LL=-84.480)
  MFCF MI-ksg  CV-LL            P=0.226  R=0.246  F1=0.235  (30.6s, 88 trials, clique=[1,32], thr=0.016, edges=124, CV-LL=-84.610)
  MFCF MI-hist CV-LL            P=0.167  R=0.070  F1=0.099  (30.1s, 836 trials, clique=[1,2], thr=0.137, edges=48, CV-LL=-84.673)
  GraphicalLassoCV              P=0.288  R=0.719  F1=0.411  (3.74s)


In [11]:
scenario_cycle()


[Cycle + chords, n>p]  p=40, n=300, |E_true|=45
  MFCF corr  HPO                P=0.838  R=0.689  F1=0.756  (30.2s, 3019 trials, clique=[1,2], thr=0.017, edges=37, gamma=0.0, EBIC=33678.3)
  MFCF MI-ksg  HPO              P=0.588  R=0.222  F1=0.323  (30.1s, 681 trials, clique=[1,21], thr=0.062, edges=17, gamma=0.0, EBIC=33740.6)
  MFCF MI-hist HPO              P=0.762  R=0.356  F1=0.485  (30.2s, 2590 trials, clique=[1,4], thr=0.029, edges=21, gamma=0.0, EBIC=33661.8)
  MFCF corr  CV-LL              P=1.000  R=0.489  F1=0.657  (30.1s, 1470 trials, clique=[1,2], thr=0.035, edges=22, CV-LL=-56.263)
  MFCF MI-ksg  CV-LL            P=0.300  R=0.333  F1=0.316  (30.2s, 229 trials, clique=[1,22], thr=0.035, edges=50, CV-LL=-56.079)
  MFCF MI-hist CV-LL            P=0.500  R=0.378  F1=0.430  (30.1s, 1250 trials, clique=[1,3], thr=0.013, edges=34, CV-LL=-56.038)
  GraphicalLassoCV              P=0.285  R=0.867  F1=0.429  (1.28s)


In [12]:
scenario_blocks_gaussian()


[Block-diag, Gaussian, n>p]  p=20, n=400, |E_true|=30
  MFCF corr  HPO                P=0.905  R=0.633  F1=0.745  (30.2s, 3571 trials, clique=[1,13], thr=0.019, edges=21, gamma=0.0, EBIC=22379.0)
  MFCF MI-ksg  HPO              P=0.846  R=0.367  F1=0.512  (30.1s, 1474 trials, clique=[1,8], thr=0.057, edges=13, gamma=0.0, EBIC=22402.6)
  MFCF MI-hist HPO              P=0.789  R=0.500  F1=0.612  (30.3s, 3030 trials, clique=[1,4], thr=0.017, edges=19, gamma=0.0, EBIC=22349.7)
  MFCF corr  CV-LL              P=0.574  R=0.900  F1=0.701  (30.1s, 2869 trials, clique=[2,15], thr=0.003, edges=47, CV-LL=-27.957)
  MFCF MI-ksg  CV-LL            P=0.348  R=0.533  F1=0.421  (30.1s, 533 trials, clique=[1,9], thr=0.005, edges=46, CV-LL=-27.861)
  MFCF MI-hist CV-LL            P=0.463  R=0.633  F1=0.535  (30.1s, 2113 trials, clique=[1,15], thr=0.001, edges=41, CV-LL=-27.842)
  GraphicalLassoCV              P=0.397  R=0.900  F1=0.551  (0.09s)


In [13]:
scenario_blocks_nongaussian()


[Non-Gaussian blocks, n>p]  p=20, n=600, |E_true|=30
  MFCF corr  HPO                P=1.000  R=0.500  F1=0.667  (30.2s, 3589 trials, clique=[1,4], thr=0.125, edges=15, gamma=0.0, EBIC=21722.0)
  MFCF MI-ksg  HPO              P=1.000  R=0.833  F1=0.909  (30.1s, 1107 trials, clique=[1,3], thr=0.152, edges=25, gamma=0.0, EBIC=16494.3)
  MFCF MI-hist HPO              P=1.000  R=0.500  F1=0.667  (30.3s, 3003 trials, clique=[1,8], thr=0.059, edges=15, gamma=0.0, EBIC=16434.5)
  MFCF corr  CV-LL              P=1.000  R=0.500  F1=0.667  (30.2s, 2844 trials, clique=[1,4], thr=0.125, edges=15, CV-LL=-18.410)
  MFCF MI-ksg  CV-LL            P=1.000  R=0.833  F1=0.909  (30.1s, 426 trials, clique=[1,4], thr=0.796, edges=25, CV-LL=-13.871)
  MFCF MI-hist CV-LL            P=1.000  R=0.533  F1=0.696  (30.2s, 2301 trials, clique=[1,16], thr=0.623, edges=16, CV-LL=-13.864)
  GraphicalLassoCV              P=0.185  R=0.967  F1=0.310  (3.24s)


In [14]:
scenario_er_highdim()


[ER precision, n<<p]  p=150, n=50, |E_true|=287
  MFCF corr  HPO                P=0.667  R=0.014  F1=0.027  (30.1s, 517 trials, clique=[1,28], thr=0.244, edges=6, gamma=0.5, EBIC=21273.6)
  MFCF MI-ksg  HPO              P=1.000  R=0.003  F1=0.007  (30.2s, 365 trials, clique=[1,21], thr=0.247, edges=1, gamma=0.5, EBIC=21036.8)
  MFCF MI-hist HPO              P=0.000  R=0.000  F1=0.000  (30.1s, 433 trials, clique=[1,6], thr=0.692, edges=0, gamma=0.5, EBIC=21041.6)
  MFCF corr  CV-LL              P=0.000  R=0.000  F1=0.000  (30.2s, 149 trials, clique=[1,35], thr=0.481, edges=0, CV-LL=-212.841)
  MFCF MI-ksg  CV-LL            P=0.000  R=0.000  F1=0.000  (30.1s, 153 trials, clique=[4,29], thr=0.655, edges=3, CV-LL=-207.740)
  MFCF MI-hist CV-LL            P=0.000  R=0.000  F1=0.000  (30.3s, 106 trials, clique=[1,14], thr=0.419, edges=0, CV-LL=-207.957)
  GraphicalLassoCV              P=0.429  R=0.021  F1=0.040  (1.54s)


In [15]:
scenario_cycle_highdim()


[Cycle + chords, n<<p]  p=80, n=30, |E_true|=88
  MFCF corr  HPO                P=0.200  R=0.011  F1=0.022  (30.1s, 1755 trials, clique=[1,23], thr=0.329, edges=5, gamma=0.5, EBIC=6802.9)
  MFCF MI-ksg  HPO              P=0.000  R=0.000  F1=0.000  (30.2s, 1299 trials, clique=[1,26], thr=0.003, edges=1, gamma=0.5, EBIC=6528.9)
  MFCF MI-hist HPO              P=0.000  R=0.000  F1=0.000  (30.2s, 1271 trials, clique=[1,22], thr=0.247, edges=0, gamma=0.5, EBIC=6532.4)
  MFCF corr  CV-LL              P=0.000  R=0.000  F1=0.000  (30.1s, 432 trials, clique=[1,7], thr=0.666, edges=0, CV-LL=-113.515)
  MFCF MI-ksg  CV-LL            P=0.000  R=0.000  F1=0.000  (30.1s, 628 trials, clique=[4,27], thr=0.469, edges=3, CV-LL=-105.682)
  MFCF MI-hist CV-LL            P=0.000  R=0.000  F1=0.000  (30.1s, 588 trials, clique=[3,28], thr=0.294, edges=1, CV-LL=-105.687)
  GraphicalLassoCV              P=0.167  R=0.023  F1=0.040  (1.80s)


In [16]:
scenario_blocks_gaussian_highdim()


[Block-diag, Gaussian, n<<p]  p=48, n=40, |E_true|=168
  MFCF corr  HPO                P=0.000  R=0.000  F1=0.000  (30.2s, 2528 trials, clique=[1,29], thr=0.273, edges=1, gamma=0.5, EBIC=5447.0)
  MFCF MI-ksg  HPO              P=0.000  R=0.000  F1=0.000  (30.2s, 2047 trials, clique=[1,13], thr=0.647, edges=0, gamma=0.5, EBIC=5416.9)
  MFCF MI-hist HPO              P=0.000  R=0.000  F1=0.000  (30.2s, 2230 trials, clique=[1,24], thr=0.318, edges=0, gamma=0.5, EBIC=5416.9)
  MFCF corr  CV-LL              P=0.000  R=0.000  F1=0.000  (30.1s, 1062 trials, clique=[1,28], thr=0.481, edges=0, CV-LL=-68.109)
  MFCF MI-ksg  CV-LL            P=0.000  R=0.000  F1=0.000  (30.1s, 985 trials, clique=[4,6], thr=0.477, edges=3, CV-LL=-66.347)
  MFCF MI-hist CV-LL            P=1.000  R=0.006  F1=0.012  (30.2s, 1154 trials, clique=[3,11], thr=0.582, edges=1, CV-LL=-66.396)
  GraphicalLassoCV              P=0.200  R=0.012  F1=0.022  (0.26s)


In [17]:
scenario_blocks_nongaussian_highdim()


[Non-Gaussian blocks, n<<p]  p=48, n=40, |E_true|=168
  MFCF corr  HPO                P=1.000  R=0.214  F1=0.353  (30.2s, 3091 trials, clique=[1,2], thr=0.623, edges=36, gamma=0.5, EBIC=271.6)
  MFCF MI-ksg  HPO              P=1.000  R=0.214  F1=0.353  (30.2s, 2395 trials, clique=[1,2], thr=0.781, edges=36, gamma=0.5, EBIC=-621.4)
  MFCF MI-hist HPO              P=1.000  R=0.387  F1=0.558  (30.2s, 2522 trials, clique=[1,3], thr=0.799, edges=65, gamma=0.5, EBIC=-622.9)
  MFCF corr  CV-LL              P=1.000  R=0.214  F1=0.353  (30.1s, 1719 trials, clique=[1,2], thr=0.437, edges=36, CV-LL=-9.028)
  MFCF MI-ksg  CV-LL            P=1.000  R=0.565  F1=0.722  (30.1s, 1248 trials, clique=[1,4], thr=0.739, edges=95, CV-LL=6.722)
  MFCF MI-hist CV-LL            P=1.000  R=0.571  F1=0.727  (30.1s, 1163 trials, clique=[1,21], thr=0.160, edges=96, CV-LL=6.241)
  GraphicalLassoCV              P=0.424  R=0.518  F1=0.466  (1.42s)


In [18]:
scenario_mi_collapse()


[MI collapse to |Pearson|]  p=12, n=10000, |E_true|=18
  similarity-matrix collapse (off-diagonal):
    pair               max-abs   mean-abs   Pearson
    KSG  vs |corr|    0.1763    0.0372    0.9847
    hist vs |corr|    0.0662    0.0154    0.9983
    hist vs KSG       0.1783    0.0401    0.9841
  MFCF corr     HPO             P=1.000  R=1.000  F1=1.000  (30.2s, 3660 trials, clique=[1,9], thr=0.365, edges=18, gamma=0.0, EBIC=265884.1)
  MFCF MI-ksg   HPO             P=1.000  R=1.000  F1=1.000  (30.3s, 151 trials, clique=[1,5], thr=0.059, edges=18, gamma=0.0, EBIC=265909.9)
  MFCF MI-hist  HPO             P=1.000  R=1.000  F1=1.000  (30.2s, 2142 trials, clique=[1,5], thr=0.059, edges=18, gamma=0.0, EBIC=265909.9)
  precision-output collapse (edge-set Jaccard; scale-invariant):
    Jaccard(E_corr, E_ksg)            = 1.0000
    Jaccard(E_corr, E_hist)           = 1.0000
    Jaccard(E_ksg,  E_hist)           = 1.0000


## Interpreting the results

**Two selection criteria, both likelihood-based and from the literature.**
- **EBIC** (`HPO` rows; Foygel & Drton 2010): penalized *in-sample* log-likelihood,
  `-2 l_n + |E| log n + 4 gamma |E| log p`. Default. `gamma="auto"` follows their
  regime guidance: `gamma=0` (classical BIC) when `n > p`, `gamma=0.5` when `p >= n`.
  Exact, because MFCF's chordal graph makes the LoGo precision the decomposable MLE
  (Lauritzen 1996).
- **CV-LL** rows: K-fold *held-out* log-likelihood — the **same outer criterion
  `GraphicalLassoCV` uses**. Included so MFCF vs GLasso is compared under an identical
  selection rule (isolating the estimator from the selection criterion).

Two guards make likelihood selection well-posed (preconditions, not heuristics):
clique size `<= n-1` (the decomposable MLE must exist) and precision condition number
`<= 1e8` (positive-definiteness — the analogue of GraphicalLasso's L1 regularisation).

**Well-posed regime (`n > p`).** MFCF beats GraphicalLassoCV. Pearson-MFCF is best on
Gaussian targets (ER, cycle, blocks); the **MI estimators win on the non-Gaussian
blocks** (MI-ksg F1 ~ 0.91 vs corr 0.67 vs GLasso 0.31) — the non-monotone-dependence
case MI is built for. The criterion-matched row confirms it (`MFCF corr CV-LL` 0.70 >
`GraphicalLassoCV` 0.55 on blocks-Gaussian, same recall, higher precision).

**Collapse scenario (sample-rich Gaussian).** Correlation, KSG-MI and histogram-MI
recover the *identical* graph (Jaccard = 1.0, F1 = 1.0) — confirming all three
similarities coincide under joint Gaussianity.

**Data-starved Gaussian regime (`n << p`: ER, cycle, blocks high-dim).** *Every* method,
MFCF and GraphicalLassoCV alike, scores ~0. This is genuine non-recoverability, not a
defect. For `blocks_gaussian_highdim` (p=48, n=40, 168 true edges) the sample covariance
is numerically singular (condition number ~3e17), so **EBIC correctly abstains**: the
in-sample likelihood gain per edge is smaller than even the basic BIC cost `log n`, so
the empty graph attains a lower (better) EBIC than every structure-recovering config —
*even at gamma=0*. The conditioning guard is **not** the cause (recovering configs have
condition number ~20-130, far below 1e8). This is EBIC's consistency / false-positive
control by design: it declines to emit edges the data cannot support, where a
density-rewarding heuristic would instead return marginally-better-than-chance guesses.